# **GUIDE**: SPEC MODULE

This notebook is designed to provide an introduction to the spec module of this simulation pipeline. This module runs PICASO atmosphere models in parallel across a parameter grid. These models are then fed into the codec module to create the spectra for the brown dwarf. This module is less involved for the user, but this notebook provides a high-level understanding of the entire process.

### **Setup**

#### **Part 1: PICASO and Virga**

To run this module, it is necessary to have the PICASO and virga reference data stored locally. Make sure to store the data in an accessible directory, since we will need the directory paths when running `spec.py`.

The instructions for download are given at the following website:

https://natashabatalha.github.io/picaso/installation.html

You will need the following data:

1. PICASO Reference Data
    * Sonora Profile Database
    * Virga Database
2. Pysynphot Stellar Data

**Directories to Store**:

These are the following directories whose paths you will need to locate. Copy the paths into the cell below; we will be inserting these paths into the `spec.py` folder itself.

In [2]:
picaso_refdata = "../picaso/reference"  # PICASO reference data
pysyn_cdbs = "../picaso/reference/stellar_spectra/grp/redcat/trds"  # pysynphot redcat data
picaso_home = "../picaso"  # PICASO home directory

sonora_profile_db = "../picaso/data/sonora_profile"  # PICASO sonora profile database
virga_directory = "../picaso/data/virga"  # PICASO virga database

Copy and paste the above files into the `Path Setup` section of `spec.py`. You should now be ready to run the module!

#### **Part 2: Importing `spec.py`**

To import the relevant functions from `spec.py`, we will follow a similar process to that from the `main.py` notebook.

**Steps**:

1. Identify and copy the path to the directory where `spec.py` is located on your machine.
2. Paste the path in the cell below (`main_path`).
3. Run the cell to import the `renderer()` function.

Copy and paste the above information into the cell below, and then run it to complete the import.

In [3]:
import sys
import os

spec_path = '../spec_module'

# Add the path to where spec.py is located
sys.path.insert(0, os.path.abspath(spec_path))

from spec import run_spec  # Import just the run_spec() function

c:\Aakanksha\UArizona\Ananconda\envs\testVtk_new\lib\site-packages\picaso\__init__.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import declare_namespace
c:\Aakanksha\UArizona\Ananconda\envs\testVtk_new\lib\site-packages\picaso\justdoit.py:54: UserWarning: Your code version is 3.2.2 but your reference data version is 3.4. For some functionality you may experience Keyword errors. Please download the newest ref version or update your code: https://github.com/natashabatalha/picaso/tree/master/reference
  warnings.warn(f"Your code version is {__version__} but your reference data version is {ref_v}. For some functionality you may experience Keyword errors. Please download the newest ref version or update your code: https://github.com/natashabatalha/picaso/tree/master/refe

### **Running the Model**

#### **Setting Atmospheric Parameters**

This module runs PICASO atmosphere models for various combinations of parameters, creating spectra for different atmosphere configurations. There are several parameters within our control with the most important ones listed below; we use these to create a parameter grid to feed into the program.

**Atmospheric Parameters to Edit**:

1. `fsed_list`: list of atmosphere cloud thickness values - *length also corresponds to number of codecs in `codec.py`!*
2. `Teff_list`: list of brown dwarf temperatures
3. `excluded_mol_list`: any molecules to be excluded from the spectra
4. `gases_list`: list of gases to be included in the spectra

In creating the parameter grid, we create a list of dictionaries: each dictionary contains a unique combination of the above parameters. This grid is created internally within the `run_spec()` function, based on the parameters passed in `config`.

Let's set up our first parameter grid. Take a look at and run the cell below!

In [4]:
fsed_list = sorted(set([1., 1.02, 1.04, 1.06, 1.08, 1.1, 1.12, 1.14, 1.16, 1.18] + 
            [1.01, 1.03, 1.05, 1.07, 1.09, 1.11, 1.13, 1.15, 1.17, 1.19]))
Teff_list = [1200]  # in Kelvin
excluded_mol_list = ['CH4']
gases_list = [['Fe', 'MgSiO3', 'Na2S']]  # NOTE THE NESTED LIST

config = {'fsed_list': fsed_list,
          'Teff_list': Teff_list,
          'excluded_mol_list': excluded_mol_list,
          'gases_list': gases_list
          }

#### **Single Configuration Model**

For each combination of parameters (configuration), `spec.py` runs a PICASO atmospheric model. While this is done in the background, here's a brief overview of the process.

##### **Step 1: Configure Atmosphere**

**What it does**: Sets up the basic atmospheric structure using information from the Sonora database

**Relevant code**: `configure_atm()`

**Outputs**:
* PICASO atmosphere object
* opacity table

##### **Step 2: Run Spectrum**

**What it does**: Computes cloud particle distribution using virga (based on configuration) and computes the spectrum with clouds included

**Relevant code**:
* `configure_cloud()`
* `run_single_model()`

**Outputs**:
* Calculated spectrum with given parameters

##### **Step 3: Outputs**

**What it does**: Writes three output files to your directory

**Outputs**:
1. Final spectrum file (.csv)
    * Wavelength + flux
2. Full model atmosphere (.nc)
    * Pressures, temperatures, opacities, etc.
3. Virga cloud output (.pkl)
    * Cloud particle size, opacity, etc.

#### **Parallel Processing Models**

To optimize time efficiency, `spec.py` processes multiple configurations (models) parallelly. This is also handled in the background and uses 1 less than your total number of CPU cores to handle the processing by default.

#### **Implementing `run_spec()`**

Running `spec.py` in a notebook is fairly straightforward, once all the configurations have been finalized: we simply need to pass our `config` dictionary as an argument to `run_spec()`.

Run the cell below to create spectra with our current test configurations!

In [5]:
run_spec(config)

Setting up parameter grid...
Total models to run: 20
Output directory: bd_grid_noCH4_20260226_012103
Running 20 models on 4 processes...
CPU count: 12
Starting parallel pool...
All models completed, writing config file...
Configuration written to bd_grid_noCH4_20260226_012103\config.txt

Results saved to bd_grid_noCH4_20260226_012103
Configuration saved to bd_grid_noCH4_20260226_012103\config.txt
Completed: 0/20 models successful
Total time: 0:00:11.586824

Example - Loading a saved model:


KeyError: 'wave'

<Figure size 3600x1800 with 0 Axes>